# OmniServe — QLoRA fine-tuning

Fine-tunes a domain SLM for invoice extraction and reports before/after accuracy.

**Runtime → Change runtime type → T4 GPU**, then Run all. ~90 minutes.

## Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
%%capture
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
# Not captured: this is the cell most likely to fail, and a silent
# clone failure would surface much later as a confusing import error.
!git clone -q https://github.com/JCHETAN26/Omni-Serve.git omniserve
%cd omniserve
!pip install -q -e .

## Config

Every knob lives here. The eval cells both read `LIMIT`, so the baseline and the
tuned run always score the same records — differing counts would make the
comparison meaningless with nothing to warn you.

In [ ]:
MODEL   = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"  # ungated 4-bit mirror
ADAPTER = "training/adapters/omniserve-slm-8b"

LIMIT   = 200    # test records to score; None for the full 500 (+~30 min)
EPOCHS  = 1      # 1 fits a free T4; use 2 on L4/A100
MAX_SEQ = 1024   # longest example is ~640 tokens
BATCH   = 8      # eval batch size

_limit = f"--limit {LIMIT}" if LIMIT else ""
print(f"{MODEL}\n{EPOCHS} epoch(s), seq {MAX_SEQ}, scoring {LIMIT or 500} records")

## Dataset

In [ ]:
!python -m data.generate_dataset --count 10000 --offline --noise 0.01
!python -m data.split_dataset

## Baseline

Measured before training, so nothing downstream can influence it.

The baseline gets `--include-schema`; the tuned model won't. It needs the schema
to know what fields to emit, and that asymmetry favours the baseline — so the
improvement comes out understated rather than inflated.

In [ ]:
!python -m benchmarks.eval_local --model {MODEL} --tag baseline \
    --include-schema --batch-size {BATCH} {_limit}

## Train

Watch the loss fall from ~1.5 toward <0.3. Flat loss means something is wrong —
stop and check the data cell output.

In [ ]:
!python -m training.train_qlora --model {MODEL} \
    --max-seq-length {MAX_SEQ} --epochs {EPOCHS} --output {ADAPTER}

## Tuned

No `--include-schema`: the model learned the schema, and omitting it saves ~400
tokens of prefill per request at serve time.

In [ ]:
!python -m benchmarks.eval_local --model {MODEL} --adapter {ADAPTER} \
    --tag tuned --batch-size {BATCH} {_limit}

## Results

In [ ]:
import json
from pathlib import Path

b = json.loads(Path("benchmarks/results/accuracy-baseline.json").read_text())
t = json.loads(Path("benchmarks/results/accuracy-tuned.json").read_text())

print(f"{'':<26}{'baseline':>10}{'tuned':>10}{'delta':>10}")
print("-" * 56)
for label, key in [
    ("Field F1", "field_f1"),
    ("Field precision", "field_precision"),
    ("Field recall", "field_recall"),
    ("Exact match rate", "exact_match_rate"),
    ("Schema validity rate", "schema_validity_rate"),
    ("Invalid JSON rate", "invalid_json_syntax_rate"),
]:
    print(f"{label:<26}{b[key]:>10.4f}{t[key]:>10.4f}{t[key] - b[key]:>+10.4f}")

This table isolates what *fine-tuning* contributed. Constrained decoding is not
active here — it lands at serve time and takes schema validity to 1.0 by itself,
so crediting it to the fine-tune would overstate both.

## Save

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!mkdir -p "/content/drive/MyDrive/omniserve"
!cp -r {ADAPTER} "/content/drive/MyDrive/omniserve/"
!cp benchmarks/results/*.json "/content/drive/MyDrive/omniserve/"
!ls -la "/content/drive/MyDrive/omniserve/"

Copy `accuracy-baseline.json` and `accuracy-tuned.json` back into the repo —
Phase 7 charts against them.